# 03 — Graph RAG

*Level 3 — Modular RAG*

## Objective
Extract (subject, relation, object) triples from the real PDF with a local LLM, build a knowledge graph with `networkx`, and answer relationship questions from it.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
for sub in ["", "graph-rag"]:
    sys.path.insert(0, str(LEVEL_DIR / sub) if sub else str(LEVEL_DIR))


In [2]:
from modular_common.pdf import load_chunks
from entity_extraction import extract_triples

chunks = load_chunks()
# The paper's title/author/affiliation block is in the first couple of chunks.
intro_chunks = list(chunks.values())[:4]

all_triples = []
for chunk in intro_chunks:
    all_triples.extend(extract_triples(chunk["text"]))

print(f"Extracted {len(all_triples)} triples:")
for t in all_triples:
    print(f"  {t['subject']} --{t['relation']}--> {t['object']}")


Extracted 28 triples:
  Ashish Vaswani --works at--> Google Brain
  Noam Shazeer --works at--> Google Brain
  Niki Parmar --works at--> Google Research
  Jakob Uszkoreit --works at--> Google Research
  Llion Jones --works at--> Google Research
  Aidan N. Gomez --works at--> University of Toronto
  Łukasz Kaiser --works at--> Google Brain
  Illia Polosukhin --works at--> Google Brain
  Transformer --generalizes well to--> English constituency parsing
  Transformer --designed and implemented--> first Transformer models
  Jakob --proposed replacing RNNs with--> self-attention
  Ashish --designed and implemented--> first Transformer models
  Transformer --achieves--> 28.4 BLEU on WMT 2014 English-to-German translation task
  Transformer --establishes a new single-model state-of-the-art--> 41.8 BLEU score on WMT 2014 English-to-French translation task
  Transformer --requires significantly less time to train than--> machine translation tasks
  Ashish --designed and implemented--> Transforme

## Build the graph and query it


In [3]:
from graph_builder import build_graph
from graph_retrieval import graph_search

graph = build_graph(all_triples)
print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges\n")

for q in ["Who is affiliated with Google Brain?", "What does the Transformer achieve?", "Tell me about the University of Toronto"]:
    print(f"Q: {q}")
    for fact in graph_search(graph, q):
        print(f"  - {fact}")
    print()


Graph: 42 nodes, 28 edges

Q: Who is affiliated with Google Brain?
  - Ashish Vaswani works at Google Brain
  - Noam Shazeer works at Google Brain
  - Łukasz Kaiser works at Google Brain
  - Illia Polosukhin works at Google Brain
  - Niki Parmar works at Google Research
  - Jakob Uszkoreit works at Google Research
  - Llion Jones works at Google Research

Q: What does the Transformer achieve?
  - Transformer generalizes well to English constituency parsing
  - Transformer designed and implemented first Transformer models
  - Transformer achieves 28.4 BLEU on WMT 2014 English-to-German translation task
  - Transformer establishes a new single-model state-of-the-art 41.8 BLEU score on WMT 2014 English-to-French translation task
  - Transformer requires significantly less time to train than machine translation tasks
  - Ashish designed and implemented Transformer models

Q: Tell me about the University of Toronto
  - Aidan N. Gomez works at University of Toronto



## What I observed

Entity extraction correctly pulled out real facts from the paper's author/affiliation block and its opening results claims — with no hand-written extraction rules, just a prompt asking for (subject, relation, object) triples.

Look closely at the extracted triples above, though: **"Jakob Uszkoreit" and "Jakob" are two separate nodes**, as are "Ashish Vaswani"/"Ashish" and "Noam Shazeer"/"Noam" — later sentences in the paper refer to authors by first name only, and this level's entity matching (case-insensitive substring, no coreference resolution) has no way to know "Jakob" and "Jakob Uszkoreit" are the same entity. That's a real, visible limitation, not a hypothetical one: a query asking "what did Jakob propose?" and one asking "what did Jakob Uszkoreit propose?" would retrieve *different* facts from this exact graph.

This also does not scale to a large corpus as-is: running extraction over every chunk of a big document means many LLM calls. Level 5+ revisits both problems — entity resolution and incremental, agentic graph construction.

## Next

[04 — Web RAG](./04_web_rag.ipynb)
